# 把科学观测写成 token，交给同一个 nanoGPT

这本 Notebook 带你把一个科学方向从头走一遍，再看怎样让程序自己改进模型。

| 节 | 做什么 | 默认方向（扩散场）在 CPU 上的用时 |
|---|---|---|
| 0 | 选方向、装依赖 | 依赖已装好时几秒 |
| 1 | 看数据：几条样本和标签分布 | 几秒 |
| 2 | 预处理：切成 token、标准化 | 几秒 |
| 3 | 搭模型：编码器、两层 nanoGPT、读出头 | 几秒 |
| 4 | 预训练：只用无标签数据，预测下一个 token | 约 20 秒 |
| 5 | 下游训练：用 32 个标签学会回答问题 | 几秒 |
| 6 | 评测：留出样本上的误差，和两条基线比 | 几秒 |
| 7 | 自动研究：让程序自己提改动、跑实验 | 看存档几秒；现场跑 2 轮约 2 到 3 分钟 |
| 8 | 星流轨道拟合：调参智能体怎样积累经验 | 约 10 秒 |

每一节开头的大写变量都可以改，改完从那一格往下重新运行。十一个方向用的是同一份代码，只在第 0 节换 `FIELD`。

可选方向：spectrum（光谱）、protein（蛋白质）、dna、rna、smiles（小分子）、crystal（晶体）、structure3d（三维结构）、weather（天气场）、field（扩散场）、image（天文图像）、text（文本）。每个方向的专业定义、单位和数据出处见仓库的 data/eleven/SOURCES.md。

## 0. 选方向，装依赖

这一格检查 PyTorch、NumPy、scikit-learn 和 Matplotlib，缺哪个装哪个。在仓库外单独打开这本 Notebook 时，它会先把仓库克隆到当前目录。全程只用 CPU。

In [ ]:
FIELD = "field"            # 换方向只改这里
SEED = 53                  # 随机种子：决定抽哪 32 个标签、模型的初始权重
N_LABELS = 32              # 下游用多少个带标签的对象。文档里的数字都是 32
THREADS = 2                # CPU 线程数。想复现文档里的数字就保持 2

import importlib.util
import subprocess
import sys
import time
from pathlib import Path

if importlib.util.find_spec("torch") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "torch",
                           "--index-url", "https://download.pytorch.org/whl/cpu"])
missing = [pip for module, pip in (("numpy", "numpy"), ("sklearn", "scikit-learn"), ("matplotlib", "matplotlib"))
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])

here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / "nanoscigpt" / "tasks" / "eleven.py").exists()), None)
if root is None:
    subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/Yu-Yang-Li/nanoSciGPT.git"])
    root = here / "nanoSciGPT"
sys.path.insert(0, str(root))

import numpy as np
import torch
torch.set_num_threads(THREADS)
from nanoscigpt.tasks import eleven, eleven_plots as plots
eleven.N_LABELS = N_LABELS

font = plots.use_chinese_font()
if font is None:
    print("没有找到中文字体，自动下载也没成功，图里的中文会显示成方框。Linux 上可以运行：sudo apt install fonts-noto-cjk")

spec = eleven.FIELDS[FIELD]
print(f"{spec['name']}（{spec['data']}数据）：{spec['question']}。标签：{spec['label']}")
print(f"一个 token = {spec['token']}")
print(f"下游任务：{spec['task']}；读出：{'冻结主干 + 线性头' if spec['readout'] == 'probe' else '全参数微调'}；"
      f"预训练：{'每档 1,600 步' if spec['budget'] == 'fixed' else '每个对象平均 32 遍'}")

## 1. 看数据

动手训练之前，先看数据长什么样。每个方向有三份数据，用途分开：

- **无标签对象**：12,800 个，只有输入，没有答案。预训练只用这一份。
- **标签池**：一两百个带答案的对象，每次从里面抽 `N_LABELS` 个（默认 32）来训练下游。
- **留出组 A**：几百个带答案的对象，不参与任何训练，只用来算最后的误差。

下面画出留出组里标签偏低、居中、偏高的三条样本，以及标签的分布。先自己看一眼：凭肉眼能不能从输入猜出标签的高低？

In [ ]:
N_EXAMPLES = 3             # 画几条样本

raw_parts, raw_task = eleven.load(FIELD)
for name, (inputs, _) in {"无标签对象": raw_parts["unlabeled"], "标签池": raw_task["pool"], "留出组 A": raw_task["A"]}.items():
    print(f"{name}：" + "，".join(f"{key} {tuple(values.shape)}" for key, values in inputs.items()))
plots.show_samples(FIELD, raw_task, N_EXAMPLES)

## 2. 预处理：把一个对象切成 token

语言模型读的是一串 token。这一步把上面的对象写成这样一串：左图是原始对象和 token 的边界，右图是模型实际读到的内容。

- 符号类输入（蛋白质、DNA、RNA、小分子、文本）：一个符号一个 token，查词表变成编号。
- 数值类输入（光谱、图像、各种场、距离矩阵）：相邻的几个数组成一个 token。
- 晶体：晶格矩阵是第一个 token，之后每个原子一个 token。

改 `ROW` 可以换一条样本看。

In [ ]:
ROW = 0                    # 看留出组里的第几条样本

plots.show_tokens(FIELD, raw_task, ROW)
eleven.show_tokens(FIELD, {name: values[ROW] for name, values in raw_task["A"][0].items()})
print("这条样本的下游标签：", round(float(raw_task["A"][1][ROW]), 4))

数值类输入还要做标准化：减去均值、除以标准差，让不同位置的数在同一个量级上。均值和标准差只从抽中的那些带标签的训练对象上算，评测数据不参与，这样留出组的信息不会漏进训练。符号类输入不需要这一步，下面改为画符号的出现次数。

In [ ]:
parts, task = eleven.normalize(FIELD, raw_parts, raw_task, SEED)
plots.show_normalization(FIELD, raw_task, task)

## 3. 搭模型

模型分三段：

1. **编码器**：把每个 token 变成一个 32 维的向量。十一个方向只有这一段不同，共三种（符号、数值块、晶胞）。
2. **两层 nanoGPT**：十一个方向结构完全相同。每个位置只能看到它前面的 token。
3. **读出头**：一个线性层，把整条序列的表示变成一个数，也就是下游任务的答案。

下面打印编码器的结构，再画出每一段有多少参数。把 `COMPARE_WITH` 换成别的方向，可以看到两个方向只有编码器和位置编码的参数个数不同。

In [ ]:
COMPARE_WITH = "protein"   # 拿哪个方向来对比参数个数

model = eleven.network(FIELD)
print(model.encoder)
print(f"主干：{len(model.blocks)} 层，每层 {model.blocks[0].attn.n_head} 个注意力头，向量 {model.blocks[0].attn.n_embd} 维，"
      f"序列最长 {model.wpe.num_embeddings} 个 token")
plots.show_model([FIELD] if COMPARE_WITH == FIELD else [FIELD, COMPARE_WITH])

## 4. 预训练：只用无标签数据

预训练的题目是“根据前面的 token 预测下一个 token”，答案就在数据自己里面，不需要任何标签。这里用不同数量的无标签对象各训练一次，看数据多了以后模型是不是学得更好。

- 曲线：训练过程中每一批数据上的损失。
- 圆点：训练结束后，在留出样本（没参加训练）上的损失。圆点比曲线末端高很多，说明模型只是记住了训练数据。

把 12800 加进 `SIZES` 可以跑到最大档，用时大约是现在的四倍。

In [ ]:
SIZES = [200, 800, 3200]   # 预训练用的无标签对象数，可以加上 12800

models = {0: eleven.pretrain(FIELD, parts, 0, SEED)}   # 0 = 不预训练，作对照
curves, pretrain_loss = {}, {}
for size in SIZES:
    start = time.perf_counter()
    curves[size] = []
    models[size] = eleven.pretrain(FIELD, parts, size, SEED, curve=curves[size])
    pretrain_loss[size] = eleven.heldout_loss(models[size], parts["A"][0])
    print(f"{size:>6} 个无标签对象：更新 {len(curves[size]):>5} 次，留出样本上的损失 {pretrain_loss[size]:.4f}，用时 {time.perf_counter() - start:.0f} 秒")
plots.show_training(curves, pretrain_loss)

## 5. 下游训练：用少量标签学会回答问题

现在才用到标签。从标签池里抽 `N_LABELS` 个对象，接在预训练好的模型后面训练读出。读出方式有两种，每个方向有自己的默认值：

- **冻结主干 + 线性头**：主干不动，只训练最后那个线性层。
- **全参数微调**：所有权重一起更新 200 步。

每个预训练规模各训练一次读出，再加一个“不预训练”的对照：同样的模型、同样的标签，只是主干没有预训练过。图里每个点是留出组的一个对象，越贴近对角线越好。

In [ ]:
READOUT = spec["readout"]  # "probe"：冻结主干 + 线性头；"finetune"：全参数微调。默认用这个方向的设置
COMPARE = (0, max(SIZES))  # 画哪两个模型的预测。0 是不预训练

downstream_error, predictions = {}, {}
for size, trained in models.items():
    predictions[size] = []
    downstream_error[size], guess_mean = eleven.downstream(FIELD, trained, task, SEED, predictions=predictions[size], readout=READOUT)
    name = "不预训练" if size == 0 else f"{size} 个对象预训练"
    print(f"{name:>14}：留出组 A 上的平均绝对误差 {downstream_error[size]:.4f}")
print(f"{'只猜平均值':>14}：{guess_mean:.4f}")
plots.show_predictions(task["A"][1], {("不预训练" if size == 0 else f"{size:,} 个对象预训练"): np.array(predictions[size]) for size in COMPARE},
                       guess_mean, spec["task"])

## 6. 评测

误差要和基线比才有意义。这里有两条基线：

- **只猜平均值**：不用任何模型，对每个对象都回答训练标签的平均值。比它还差，说明模型没学到东西。
- **不预训练**：检验预训练有没有帮上忙。

左图是下游误差随预训练数据量的变化，和两条基线画在一起。右图是预训练损失。要看的是两条曲线是不是随数据量一起下降，以及下游误差落在两条基线的哪一边。

这里只跑了一个种子、到 3,200 档，曲线会有起伏。五个种子、四档的结果见仓库的 docs/pretraining-scale/shared-checkpoints/README.md。

In [ ]:
plots.show_scaling(pretrain_loss, downstream_error, guess_mean, f"{spec['name']} · {spec['question']}", spec["task"])
best = min(SIZES, key=downstream_error.get)
print(f"最好的一档是 {best:,} 个对象：误差 {downstream_error[best]:.4g}，"
      f"比不预训练{'低' if downstream_error[best] < downstream_error[0] else '高'} {abs(1 - downstream_error[best] / downstream_error[0]):.0%}，"
      f"比只猜平均值{'低' if downstream_error[best] < guess_mean else '高'} {abs(1 - downstream_error[best] / guess_mean):.0%}。")

## 7. 自动研究：让程序自己提改动、跑实验

前面每个设置都是人定的。这一节把“改设置、跑实验、看结果、再改”交给程序。十一个方向用同一套规则：

- **不能改**：两层 nanoGPT 主干。
- **可以改**：相邻两个 token 要不要并成一个（`merge`）、预训练的题目（预测下一个、隔一个预测、先遮住一部分再预测）、预训练步数和学习率、读出方式和它的步数、学习率、岭回归强度、表示的汇总方式。
- **怎么打分**：每个配置在开发集上评分。开发集是标签池里没被抽去训练的对象。留出组 A 只在搜索结束后用一次，评开发集上最好的那个配置。
- **两种提改动的办法**：DSK（一个大语言模型）读完此前每一轮的结果再提下一个配置；随机搜索在同样的范围里随便抽。两边轮数相同。

先看存档的结果：每个方向 DSK 和随机搜索各跑 2 次，每次 8 轮。跑不赢随机搜索的方向照实标出。

In [ ]:
from autoresearch import common_search as search, search_plots

runs = search.archived(FIELD)
search_plots.show_search(FIELD, runs)
for run in runs:
    if run["method"] == "model" and run["repeat"] == 1:
        print("DSK 第 1 次运行每一轮的理由：")
        for trial in run["trials"][1:]:
            print(f"  第 {trial['round']} 轮，开发集误差 {trial['dev']:.4f}：{trial['reason'][:90]}")

十一个方向放在一起看：

In [ ]:
search_plots.show_all_fields()

下面现场跑一次。默认用随机搜索，不需要任何密钥。想让 DSK 来提改动，先在启动 Jupyter 之前设置环境变量 `SCNET_API_BASE`、`SCNET_API_KEY`、`SCNET_MODEL`，再把 `METHOD` 改成 "model"。密钥只从环境变量读，不要写进 Notebook。

默认只跑 2 轮、1 个种子，为的是几分钟内看完流程。这么少的轮数很可能找不到比起点更好的配置，结果的波动也比存档大。想认真搜索就把轮数和种子调到存档的设置，用时会长到几十分钟。运行记录存在仓库的 `out/` 目录下，同一格重新运行会接着上次的进度；换 `DEV_SEEDS` 之前请先删掉那里的记录。

In [ ]:
METHOD = "random"          # "random" 随机搜索；"model" 让 DSK 提改动（需要环境变量里有密钥）
ROUNDS = 2                 # 轮数。存档用的是 8
DEV_SEEDS = (53,)          # 每个配置用几个种子打分。存档用的是 (53, 71, 89)

journal = search.run(FIELD, METHOD, root / "out" / f"search-{N_LABELS}labels", rounds=ROUNDS, dev_seeds=DEV_SEEDS, confirm_seeds=DEV_SEEDS)
search_plots.show_search(FIELD, [journal])

## 8. 星流轨道拟合：调参智能体怎样积累经验

上一节的程序每次都从零开始，上一次搜索学到的东西不会留给下一次。这一节换一个任务，看“把经验留下来”是什么样子。

任务是拟合银河系里一条恒星流（GD-1）的轨道：给定一组观测，调 7 个参数，让算出来的轨道穿过观测点。智能体是 DSK，一次拟合只能试 40 组参数。实验分两层：

- **一次拟合之内**：DSK 每交 10 组参数，看到拟合得好不好，再交下一批。
- **多次拟合之间**：DSK 把调参心得写成一份一千多字的经验。每轮先带着经验拟合两个训练场景，对照答案改写经验，然后锁定，拿去拟合 8 个从没见过的场景。共 8 轮。

这一节的图全部来自存档的参数，轨道在你的电脑上重新积分，不需要密钥。先看一个没见过的场景：每一轮的经验各拟合一次，找到的最好轨道叠在观测点上。

In [ ]:
SCENARIO = 9005            # 盲测场景，可选 9001 到 9008
SHOW_ROUNDS = (0, 1, 2, 5, 8)   # 画哪几轮的经验。0 是空经验

from autoresearch import stream_plots

stream_plots.show_orbits(SCENARIO, SHOW_ROUNDS)

一次拟合之内发生了什么：上排是空经验，下排是第 8 轮的经验，颜色从深到浅是第 1 到第 4 次提交之后的最好轨道。

In [ ]:
stream_plots.show_one_fit(SCENARIO, rounds=(0, 8))

经验是怎么被改写的。

- 第一张图：九条有代表性的规则，在每一轮的经验里有没有。能看到有的规则一开始就有并一直留着，有的被证据推翻后删掉（“围绕一个中心点撒点”），有的后来才加进来。
- 第二张图：经验的长度、每次改写保留了上一轮多少文字、每一轮的规则注明依据的是哪几轮的训练场景。

图下面是 DSK 自己写的每轮改动说明。想读某一轮经验的全文，把 `READ_ROUND` 改成那一轮的编号。

In [ ]:
READ_ROUND = None          # 填 1 到 8 的数字，打印那一轮经验的全文

stream_plots.show_skill_changes()
if READ_ROUND:
    print(stream_plots.skill_text(READ_ROUND))

最后看成绩。纵轴是在没见过的场景上，找到的最好轨道离答案还有多远，越低越好。

读这张图要注意三件事：带着经验比空经验好；最好的是第 5 轮，之后三轮又慢慢变差，没有一轮比一轮好下去；和同样预算的 Nelder–Mead（一种不用模型的经典优化方法）相比，优势不明显。经验的完整文本、实验规则和后续复验见仓库的 docs/stream-skill.md。

In [ ]:
stream_plots.show_blind_scores()

## 接下来可以做什么

- 回到第 0 节换一个 `FIELD`，从头再跑一遍，比较两个方向的图。
- 第 0 节把 `N_LABELS` 改成 8 或 128，从头再跑：标签更少时，预训练的帮助是更大还是更小？
- 第 5 节换一种 `READOUT`：冻结主干和全参数微调，哪一种在这个方向上更好？
- 第 7 节把 `ROUNDS` 和 `DEV_SEEDS` 调到存档的设置，看你的随机搜索和存档里的差多少。
- 命令行里做同样的事：`python -m nanoscigpt.tasks.eleven --field field` 和 `python -m autoresearch.common_search --field field --method random`。